# Episode VI — Alive
## Building a Two-Layer Neural Network from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pchambet/Deep-Learning-from-Scratch/blob/main/notebooks/06_alive.ipynb)

> *You've done the math. Now watch it breathe.*

This is the companion notebook for **Episode VI** of the *Deep Learning from Scratch* series.

In [Episode V — The Rise of Intelligence](https://github.com/Pchambet/Deep-Learning-from-Scratch/blob/main/pdf/The%20Rise%20of%20Intelligence.pdf), we derived every equation for a two-layer neural network.

Now we turn those equations into working Python code.

**What you'll build:**
1. Initialization — giving the network random parameters
2. Forward propagation — computing predictions
3. Backpropagation — computing gradients (6 lines of code)
4. Training loop — putting it all together
5. Decision boundary visualization
6. Real image classification (cats vs dogs)

---

## Setup

> **Running on Colab?** Just click the badge above — the next cell handles everything automatically (cloning the repo, installing dependencies, loading the data). Nothing to uncomment.

In [ ]:
import os, sys

# ── Auto-setup for Google Colab ──────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.isdir('Deep-Learning-from-Scratch'):
        !git clone --depth 1 https://github.com/Pchambet/Deep-Learning-from-Scratch.git
    os.chdir('Deep-Learning-from-Scratch')
    !pip install -q h5py tqdm scikit-learn matplotlib

# ── Imports ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.metrics import accuracy_score
from tqdm import tqdm

%matplotlib inline
plt.style.use('default')
np.random.seed(0)

print("✓ Setup complete — you're ready to go.")

---
## 1. The Dataset — A Problem Worth Solving

You're a data scientist on an expedition. You've discovered unknown plant species — some toxic, some safe.
You measure two features: **leaf length** and **leaf width**.

A single neuron can only draw a straight line. But the data isn't linearly separable.

We need a neural network.

In [ ]:
# Generate the dataset
X, y = make_circles(n_samples=100, noise=0.1, factor=0.3, random_state=0)
X = X.T                            # shape: (2, 100)
y = y.reshape((1, y.shape[0]))     # shape: (1, 100)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Visualize
plt.figure(figsize=(6, 6))
plt.scatter(X[0, :], X[1, :], c=y.flatten(), cmap='RdBu', edgecolors='k', s=40)
plt.xlabel('$x_1$ (leaf length)')
plt.ylabel('$x_2$ (leaf width)')
plt.title('Toxic vs Non-Toxic Plants')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

No straight line can separate these two classes. The blue points form a ring around the red cluster.

This is exactly the kind of problem that requires a **neural network**.

---
## 2. Initialization — Giving the Network a Body

Our network has:
- **2 inputs** ($n_0 = 2$): leaf length and width
- **32 hidden neurons** ($n_1 = 32$): we choose this
- **1 output** ($n_2 = 1$): binary classification

We initialize weights **randomly** to break symmetry. If all weights started at zero, every neuron would compute the same thing — forever.

In [ ]:
def initialization(n0, n1, n2):
    """Initialize random parameters for a 2-layer network."""
    W1 = np.random.randn(n1, n0)   # (32, 2)
    b1 = np.random.randn(n1, 1)    # (32, 1)
    W2 = np.random.randn(n2, n1)   # (1, 32)
    b2 = np.random.randn(n2, 1)    # (1, 1)

    parameters = {
        'W1': W1, 'b1': b1,
        'W2': W2, 'b2': b2
    }
    return parameters

# Test it
params = initialization(2, 32, 1)
for key, val in params.items():
    print(f"{key}: shape {val.shape}")

**Checkpoint — Dimensions:**

| Parameter | Shape | Why |
|-----------|-------|-----|
| $W_1$ | $(n_1, n_0)$ | $n_1$ neurons, each with $n_0$ weights |
| $b_1$ | $(n_1, 1)$ | one bias per neuron |
| $W_2$ | $(n_2, n_1)$ | $n_2$ outputs, connected to $n_1$ neurons |
| $b_2$ | $(n_2, 1)$ | one bias per output |

---
## 3. Forward Propagation — The First Breath

The same structure as Episode V — we use **sigmoid** for both layers here (Ep. V uses *tanh* in the hidden layer; the backprop pattern is identical, only the activation derivative changes):

**Layer 1:** $Z_1 = W_1 \cdot X + b_1$, $A_1 = \sigma(Z_1)$

**Layer 2:** $Z_2 = W_2 \cdot A_1 + b_2$, $A_2 = \sigma(Z_2)$

$A_2$ is the network's prediction.

In [ ]:
def forward_propagation(X, parameters):
    """Compute forward pass through the network."""
    W1 = parameters['W1']
    b1 = parameters['b1']
    W2 = parameters['W2']
    b2 = parameters['b2']

    # Layer 1
    Z1 = W1.dot(X) + b1            # Linear transformation
    A1 = 1 / (1 + np.exp(-Z1))     # Sigmoid activation

    # Layer 2 (output)
    Z2 = W2.dot(A1) + b2           # Linear transformation
    A2 = 1 / (1 + np.exp(-Z2))     # Sigmoid activation

    activations = {'A1': A1, 'A2': A2}
    return activations

# Test forward propagation
activations = forward_propagation(X, params)
print(f"A1 shape: {activations['A1'].shape}")  # (32, 100)
print(f"A2 shape: {activations['A2'].shape}")  # (1, 100)

6 lines of computation. Same structure as Episode V — **now alive**.

---
## 4. Backpropagation — Learning from Mistakes

### Cost Function
We measure error with **log-loss** (binary cross-entropy):

$$\mathcal{L} = -\frac{1}{m} \sum \left[ y \log(A_2) + (1-y) \log(1-A_2) \right]$$

In [ ]:
def log_loss(A, y):
    """Binary cross-entropy loss."""
    m = y.shape[1]
    epsilon = 1e-15  # prevent log(0)
    return -1/m * np.sum(
        y * np.log(A + epsilon) +
        (1 - y) * np.log(1 - A + epsilon)
    )

### Gradient Equations → Code

Same pattern as Episode V (here with sigmoid: derivative $\sigma'(z) = \sigma(1-\sigma)$):

**Output layer:**
$$dZ_2 = A_2 - y \qquad dW_2 = \frac{1}{m} dZ_2 \cdot A_1^\top \qquad db_2 = \frac{1}{m} \sum dZ_2$$

**Hidden layer:**
$$dZ_1 = W_2^\top \cdot dZ_2 \circ A_1 \circ (1 - A_1) \qquad dW_1 = \frac{1}{m} dZ_1 \cdot X^\top \qquad db_1 = \frac{1}{m} \sum dZ_1$$

*This is why you did the math.*

In [ ]:
def back_propagation(X, y, parameters, activations):
    """Compute gradients via backpropagation."""
    A1 = activations['A1']
    A2 = activations['A2']
    W2 = parameters['W2']
    m = y.shape[1]

    # Output layer gradients
    dZ2 = A2 - y
    dW2 = 1/m * dZ2.dot(A1.T)
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)

    # Hidden layer gradients
    dZ1 = np.dot(W2.T, dZ2) * A1 * (1 - A1)
    dW1 = 1/m * dZ1.dot(X.T)
    db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {
        'dW1': dW1, 'db1': db1,
        'dW2': dW2, 'db2': db2
    }
    return gradients

**6 lines of gradient computation.** That's the entire backpropagation.

> Notice: `*` is element-wise (Hadamard product). `np.dot()` is matrix multiplication. Getting this wrong is a classic bug.

---
## 5. Update — One Step Closer

Gradient descent: move each parameter in the direction that reduces the loss.

$$W \leftarrow W - \alpha \cdot dW \qquad b \leftarrow b - \alpha \cdot db$$

In [ ]:
def update(gradients, parameters, learning_rate):
    """Update parameters using gradient descent."""
    parameters = {
        'W1': parameters['W1'] - learning_rate * gradients['dW1'],
        'b1': parameters['b1'] - learning_rate * gradients['db1'],
        'W2': parameters['W2'] - learning_rate * gradients['dW2'],
        'b2': parameters['b2'] - learning_rate * gradients['db2']
    }
    return parameters

---
## 6. Prediction — Making Decisions

If $A_2 \geq 0.5$, predict class 1. Otherwise, class 0.

In [ ]:
def predict(X, parameters):
    """Binary prediction: A2 >= 0.5 -> class 1."""
    activations = forward_propagation(X, parameters)
    A2 = activations['A2']
    return A2 >= 0.5

---
## 7. The Training Loop — Putting It All Together

At each epoch:
1. **Forward** — compute predictions
2. **Cost** — measure the error
3. **Backprop** — compute gradients
4. **Update** — adjust parameters

In [ ]:
def neural_network(X, y, n1=32, learning_rate=0.1, n_epochs=1000):
    """Train a 2-layer neural network from scratch."""
    n0 = X.shape[0]   # input dimension
    n2 = y.shape[0]   # output dimension

    # Initialize
    np.random.seed(0)
    parameters = initialization(n0, n1, n2)

    # History
    train_loss = []
    train_acc = []

    for i in tqdm(range(n_epochs)):
        # 1. Forward propagation
        activations = forward_propagation(X, parameters)
        A2 = activations['A2']

        # 2. Compute cost
        loss = log_loss(A2, y)
        train_loss.append(loss)

        # 3. Compute accuracy
        y_pred = (A2 >= 0.5).astype(float)
        acc = accuracy_score(y.flatten(), y_pred.flatten())
        train_acc.append(acc)

        # 4. Backpropagation
        gradients = back_propagation(X, y, parameters, activations)

        # 5. Update parameters
        parameters = update(gradients, parameters, learning_rate)

    return parameters, train_loss, train_acc

### Let's train!

In [ ]:
parameters, train_loss, train_acc = neural_network(
    X, y, n1=32, learning_rate=0.1, n_epochs=1000
)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_loss, color='#29629E')
ax1.set_title('Training Loss', fontweight='bold')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Log-Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(train_acc, color='#2E8B57')
ax2.set_title('Training Accuracy', fontweight='bold')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final loss: {train_loss[-1]:.4f}")
print(f"Final accuracy: {train_acc[-1]:.1%}")

**The network learned.** From random weights to near-perfect classification — in seconds.

---
## 8. Decision Boundary — Seeing Intelligence

The decision boundary shows *where* the network switches from predicting class 0 to class 1.

A single neuron draws a line. A network draws **curves**.

In [ ]:
def plot_decision_boundary(X, y, parameters, title="Decision Boundary"):
    """Plot the network's decision boundary."""
    x_min, x_max = X[0].min() - 1, X[0].max() + 1
    y_min, y_max = X[1].min() - 1, X[1].max() + 1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, 0.01),
        np.arange(y_min, y_max, 0.01)
    )
    grid = np.c_[xx.ravel(), yy.ravel()].T  # (2, N)
    Z = predict(grid, parameters)
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(6, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    plt.scatter(X[0], X[1], c=y.flatten(), cmap='RdBu', edgecolors='k', s=40)
    plt.title(title, fontweight='bold')
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.axis('equal')
    plt.grid(True, alpha=0.2)
    plt.show()

plot_decision_boundary(X, y, parameters, "32 neurons — 1000 epochs")

### The effect of neuron count

Let's see what happens with different numbers of hidden neurons (6 trainings × 1000 epochs ≈ 2–3 min):

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
neurons_list = [1, 2, 4, 8, 16, 32]

for idx, n1 in enumerate(neurons_list):
    ax = axes[idx // 3, idx % 3]

    # Train
    params_n, _, _ = neural_network(X, y, n1=n1, learning_rate=0.1, n_epochs=1000)

    # Plot
    x_min, x_max = X[0].min() - 1, X[0].max() + 1
    y_min, y_max = X[1].min() - 1, X[1].max() + 1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, 0.02),
        np.arange(y_min, y_max, 0.02)
    )
    grid = np.c_[xx.ravel(), yy.ravel()].T
    Z = predict(grid, params_n).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[0], X[1], c=y.flatten(), cmap='RdBu', edgecolors='k', s=20)

    y_pred = predict(X, params_n)
    acc = accuracy_score(y.flatten(), y_pred.flatten())
    ax.set_title(f'{n1} neuron{"s" if n1 > 1 else ""} \u2014 acc: {acc:.0%}', fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Decision Boundaries by Neuron Count', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**
- **1 neuron** → linear boundary (= logistic regression)
- **2 neurons** → slightly curved, not enough
- **4+ neurons** → nonlinear boundaries that capture the circular structure
- **32 neurons** → near-perfect separation

This is what intelligence looks like: the ability to draw the right boundary, automatically, from data alone.

---
## 9. Real Data — Cats vs Dogs

Let's test our network on real images.

> The data loads automatically — whether you're on Colab or running locally.

In [ ]:
# Load the cats vs dogs dataset
sys.path.insert(0, os.getcwd() if os.path.isdir('src') else os.path.dirname(os.getcwd()))
from src.utilities import load_data

X_train, y_train, X_test, y_test = load_data()
print(f"Raw shapes: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"            X_test {X_test.shape}, y_test {y_test.shape}")

In [ ]:
# Show some images
plt.figure(figsize=(12, 3))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title('Cat' if y_train.flat[i] == 0 else 'Dog', fontsize=9)
    plt.axis('off')
plt.suptitle('Training samples', fontweight='bold')
plt.tight_layout()
plt.show()

### Data Preprocessing

Each 64×64 image becomes a vector of 4096 pixels, normalized to [0, 1].

In [ ]:
# Reshape: (N, 64, 64) -> (4096, N)
X_train_flat = X_train.reshape(X_train.shape[0], -1).T / 255.
X_test_flat = X_test.reshape(X_test.shape[0], -1).T / 255.

# Labels: (1, N)
y_train_r = y_train.reshape(1, -1)
y_test_r = y_test.reshape(1, -1)

print(f"X_train_flat: {X_train_flat.shape}")
print(f"X_test_flat:  {X_test_flat.shape}")
print(f"y_train:      {y_train_r.shape}")
print(f"y_test:       {y_test_r.shape}")

### Training with test tracking

*10,000 epochs ≈ 1–2 min — we need many epochs to clearly see overfitting.*

In [ ]:
def neural_network_with_test(X_train, y_train, X_test, y_test,
                             n1=32, learning_rate=0.1, n_epochs=1000):
    """Train with train + test tracking."""
    n0 = X_train.shape[0]
    n2 = y_train.shape[0]

    np.random.seed(0)
    parameters = initialization(n0, n1, n2)

    train_loss_h, test_loss_h = [], []
    train_acc_h, test_acc_h = [], []

    for i in tqdm(range(n_epochs)):
        # Forward (train)
        act_train = forward_propagation(X_train, parameters)
        train_loss_h.append(log_loss(act_train['A2'], y_train))

        # Forward (test)
        act_test = forward_propagation(X_test, parameters)
        test_loss_h.append(log_loss(act_test['A2'], y_test))

        # Accuracy (reuse activations already computed)
        train_acc_h.append(accuracy_score(
            y_train.flatten(), (act_train['A2'] >= 0.5).astype(float).flatten()))
        test_acc_h.append(accuracy_score(
            y_test.flatten(), (act_test['A2'] >= 0.5).astype(float).flatten()))

        # Backprop + Update (train only)
        gradients = back_propagation(X_train, y_train, parameters, act_train)
        parameters = update(gradients, parameters, learning_rate)

    return parameters, train_loss_h, test_loss_h, train_acc_h, test_acc_h

In [ ]:
params_img, train_loss, test_loss, train_acc, test_acc = neural_network_with_test(
    X_train_flat, y_train_r, X_test_flat, y_test_r,
    n1=64, learning_rate=0.1, n_epochs=10000
)

In [ ]:
# Plot train vs test
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_loss, label='Train', color='#29629E')
ax1.plot(test_loss, label='Test', color='#CC7722', linestyle='--')
ax1.set_title('Loss', fontweight='bold')
ax1.set_xlabel('Epochs')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_acc, label='Train', color='#29629E')
ax2.plot(test_acc, label='Test', color='#CC7722', linestyle='--')
ax2.set_title('Accuracy', fontweight='bold')
ax2.set_xlabel('Epochs')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Cats vs Dogs \u2014 64 neurons, lr=0.1', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Final train accuracy: {train_acc[-1]:.1%}")
print(f"Final test accuracy:  {test_acc[-1]:.1%}")

## 10. Overfitting — The First Warning

Look at the curves:
- **Training loss** keeps decreasing ✓
- **Test loss** starts **increasing** after some point ✗
- **Training accuracy** climbs toward 100% ✓
- **Test accuracy** plateaus or drops ✗

> **The model is memorizing the training data instead of learning general patterns.**

This is **overfitting**.

**Why?**
- Our network has ~262K parameters ($4096 \times 64 + 64 + 64 + 1$)
- But we only have 1,000 training images
- No regularization, no data augmentation

> *"Is this a problem with the data, the model, or the training?"*

Here, the answer is clear: **not enough data** for this many parameters.

---
## What You've Built

Starting from nothing but equations, you built:

1. **Initialization** — random parameters, controlled dimensions
2. **Forward propagation** — input to prediction in 6 lines
3. **Cost function** — log-loss to measure errors
4. **Backpropagation** — 6 lines of gradient computation
5. **Update** — 4 subtractions (gradient descent)
6. **Training loop** — forward → cost → backward → update, repeat
7. **Decision boundary** — see what the network learned
8. **Image classification** — real data, real results

**No framework. No library hiding the math. Just NumPy and understanding.**

---
## What's Next

Our network has exactly 2 layers. What if we wanted 3? Or 5? Or 50?

> *What if we could build a network with **any number of layers**, **any architecture** we want?*

That's not hypothetical. **That's the next episode.**

The code barely changes. The understanding you've built here carries forward.

*Meet me in the next episode.*

---

## Share this notebook

Found this useful? Share it with someone who's learning deep learning.

| | |
|---|---|
| **Run it** | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pchambet/Deep-Learning-from-Scratch/blob/main/notebooks/06_alive.ipynb) |
| **Star the repo** | [github.com/Pchambet/Deep-Learning-from-Scratch](https://github.com/Pchambet/Deep-Learning-from-Scratch) |
| **Follow the series** | [Pierre Chambet on LinkedIn](https://www.linkedin.com/in/pierre-chambet/) |

*Deep Learning from Scratch — by Pierre Chambet*